In [ ]:
import sys
from pathlib import Path
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

project_root = Path.cwd()
while project_root.name != 'python' and project_root.parent != project_root:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Downloading the data
#### Source: _Yahoo Finance_

In [ ]:
from data import download_tickers_history

# set the date range for the historic data
start_date = datetime(year=2020, month=1, day=1)
end_date = datetime(year=2025, month=12, day=31)
history = download_tickers_history(start_date, end_date, ['NVDA']);

nvda = history.NVDA;


## Basic data analytics

#### Returns
Working with raw prices (Close) is impossible in statistics, as they are non-stationary (the trend may be up or down, and the mathematical expectation may change over time). The first thing to do is to calculate the logarithmic return.

$$R_t = \ln(P_t / P_{t-1}) = \ln(P_t) - \ln(P_{t-1}),$$

where $t$ - time (day)

In [ ]:
prices = nvda['Close'].to_numpy()
log_returns = np.diff(np.log(prices))

mu = np.mean(log_returns)
sigma = np.std(log_returns)

# data visualization
plt.figure(figsize=(10, 6))

plt.hist(
    log_returns, 
    density=True,
    bins=100,
    linewidth=0.5,
    edgecolor='w',
    label='Log Returns'
);

x = np.linspace(mu - 4*sigma, mu + 4*sigma, 500)
gauss = norm.pdf(x, loc=mu, scale=sigma)

plt.plot(
    x,
    gauss, 
    color='#d62728', 
    linewidth=2.5, 
    linestyle='--', 
    label=f'Gauss distribution\n($\mu={mu:.4f}$, $\sigma={sigma:.4f}$)'
)

plt.xlabel('Logarithmic Returns ($R_t$)', fontsize=12)
plt.ylabel('Distribution Density', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

plt.show();


### Find the distribution

We're gonna use some quantitative criteria (Skewness and Excess Kurtosis), and also test test the hypothesis that the observed sample of returns follows a normal distribution using two different criterias:
* Pirson (Сhi-square)
* Shapiro-Wilk test

In [ ]:
confidence_lvl = 0.95
alpha = 1 - confidence_lvl

In [ ]:
from scipy.stats import skew, kurtosis

skewness = skew(log_returns)
excess_kurtosis = kurtosis(log_returns)

print("Skewness:", skewness)
print("Excess Kurtosis:", excess_kurtosis)

if skewness in [0.1, 0.5] or excess_kurtosis > 0:
    print("The distribution is likely NOT Normal.")
else:
    print("Looks like it is Normal distribution.")

#### Chi-square test for narmality (Pearson criteria)

In [ ]:
from scipy.stats import chisquare

# Normal with mu=0.0023 and sigma=0.0335
n = len(log_returns)

# split into intervals (bins)
num_bins = 20
observed_freq, bin_edges = np.histogram(log_returns, bins=num_bins)

# generate random 
cdf_values = norm.cdf(bin_edges, loc=mu, scale=sigma) # Gauss CDF
expected_prob = np.diff(cdf_values) # probability of falling into the i-th interval
expected_freq = expected_prob * n

expected_freq = expected_freq * (observed_freq.sum() / expected_freq.sum())

chi2_stat, p_value = chisquare(f_obs=observed_freq, f_exp=expected_freq, ddof=2)

print(f"Criterion statistics: {chi2_stat:.3f}")
print(f"P-value: {p_value:.5e}")

if p_value < alpha:
    print("Hypothesis H0 is rejected: The distribution IS NOT Normal.")
else:
    print("Hypothesis H0 IS NOT rejected: Looks like it is Normal distribution.")


#### Shapiro-Wilk test for normality

In [ ]:
from scipy.stats import shapiro

stat, p_value = shapiro(log_returns)
print(f"p-value: {p_value}")

if p_value < alpha:
    print("Hypothesis H0 is rejected: The distribution IS NOT Normal.")
else:
    print("Hypothesis H0 IS NOT rejected: Looks like it is Normal distribution.")



### Prices log-normal distribution

In [ ]:
from scipy.stats import lognorm

log_prices = np.log(prices)
mu = np.mean(log_prices)
sigma = np.std(log_prices)
scale = np.exp(mu)

# data visualization
plt.figure(figsize=(10, 6))

plt.hist(
    prices, 
    density=True,
    bins=100,
    linewidth=0.5,
    edgecolor='w',
    label='Prices'
);

x = np.linspace(np.min(prices), np.max(prices), 500)
# build log-norm distr pdf with the mu and sigma calculated from the log_prices,
# assuming the prices have log-norm distr
log_norm = lognorm.pdf(x, s=sigma, scale=scale)

plt.plot(
    x,
    log_norm, 
    color="#6dd627", 
    linewidth=2.5, 
    linestyle='--', 
    label=f'Log-normal distribution\n($\mu={mu:.4f}$, $\sigma={sigma:.4f}$)'
)

plt.xlabel('Close Prices ($P_t$)', fontsize=12)
plt.ylabel('Distribution Density', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

plt.show();


In [ ]:
from scipy.stats import chisquare

# Log-normal with mu and sigma
n = len(prices)
log_prices = np.log(prices)
mu = np.mean(log_prices)
sigma = np.std(log_prices)
scale = np.exp(mu)

# split into intervals (bins)
num_bins = 20
observed_freq, bin_edges = np.histogram(prices, bins=num_bins)

cdf_values = lognorm.cdf(bin_edges, s=sigma, scale=scale) # log-normal CDF
expected_prob = np.diff(cdf_values) # probability of falling into the i-th interval
expected_freq = expected_prob * n

expected_freq = expected_freq * (observed_freq.sum() / expected_freq.sum())

chi2_stat, p_value = chisquare(f_obs=observed_freq, f_exp=expected_freq, ddof=2)

print(f"Criterion statistics: {chi2_stat:.3f}")
print(f"P-value: {p_value:.5e}")

if p_value < alpha:
    print("Hypothesis H0 is rejected: The distribution IS NOT Log-Normal.")
else:
    print("Hypothesis H0 IS NOT rejected: Looks like it is Log-Normal distribution.")